# Customer Churn Prediction
End-to-end ML pipeline: load data → clean → feature engineer → train 5 models → tune → save

## Prerequisites

### Python & dependencies

This notebook requires **Python 3.9+**. Install all dependencies from the **project root**:

```bash
pip install -r requirements.txt
```

Key packages used in this notebook: `pandas`, `numpy`, `scikit-learn`, `xgboost`,
`matplotlib`, `seaborn`.

---

### Dataset

Download the **Bank Customer Churn Prediction** dataset from Kaggle:

> <https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction>

Two accepted locations (the setup cell below checks both in order):

| Location | Filename | Notes |
|---|---|---|
| `data/churn.csv` | Renamed file | **Preferred** — matches production training scripts |
| `Churn_Modelling.csv` | Original Kaggle filename | Fallback for quick local runs |

---

### Where to run

Launch Jupyter from the **project root** (the folder containing `api.py` and `requirements.txt`),
not from inside `notebooks/`. This ensures relative paths for data loading and model saving
resolve correctly.

```bash
cd /path/to/customer-churn-prediction   # project root
jupyter lab notebooks/churn.ipynb
```

---

### Model output

Section 10 saves models to `models/bank/` (the path the production API reads from).
This notebook trains **5 models** (XGBoost, Random Forest, Decision Tree, SVM, KNN).
The production API expects **7** — for the full set including Gradient Boosting and
Stacking, plus `schema.json`, run `training/train_bank.py` instead.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
import sys, pathlib

print(f'Python {sys.version}')

# Resolve dataset path: prefer project-standard location, fall back to raw Kaggle name
_candidates = [pathlib.Path('data/churn.csv'), pathlib.Path('Churn_Modelling.csv')]
DATA_CSV = next((p for p in _candidates if p.exists()), None)
if DATA_CSV is None:
    raise FileNotFoundError(
        'Dataset not found. Expected one of:\n'
        + '\n'.join(f'  {p.resolve()}' for p in _candidates)
        + '\nSee the Prerequisites section above for download instructions.'
    )
print(f'Dataset : {DATA_CSV.resolve()} ({DATA_CSV.stat().st_size:,} bytes)')

# Ensure model output directory exists
MODELS_DIR = pathlib.Path('models/bank')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Models  : {MODELS_DIR.resolve()}/')

## 2. Load Data

In [ ]:
# DATA_CSV is resolved above (data/churn.csv or Churn_Modelling.csv fallback)
df = pd.read_csv(DATA_CSV)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nChurn distribution:\n', df['Exited'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn distribution
df['Exited'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)
axes[0].set_ylabel('Count')

# Churn by Geography
df.groupby('Geography')['Exited'].mean().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Churn Rate by Geography')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ['Age', 'Balance', 'CreditScore']):
    df[df['Exited'] == 0][col].hist(ax=ax, alpha=0.6, label='Retained', bins=30, color='steelblue')
    df[df['Exited'] == 1][col].hist(ax=ax, alpha=0.6, label='Churned', bins=30, color='salmon')
    ax.set_title(f'{col} Distribution')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric cols only)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 4. Data Cleaning & Feature Engineering

In [ ]:
# Drop columns not useful for prediction
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# Encode categorical variables
le_gender = LabelEncoder()
le_geo = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender'])        # Female=0, Male=1
df['Geography'] = le_geo.fit_transform(df['Geography'])    # France=0, Germany=1, Spain=2

print('Gender classes:', le_gender.classes_)
print('Geography classes:', le_geo.classes_)
df.head()

In [ ]:
# Feature engineering
df['BalanceSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['TenureAgeRatio'] = df['Tenure'] / (df['Age'] + 1)
df['CreditScorePerAge'] = df['CreditScore'] / (df['Age'] + 1)

print('New features added. Shape:', df.shape)

## 5. Train/Test Split & Scaling

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 6. Train 5 ML Models

In [ ]:
models = {
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier()
}

results = {}

for name, model in models.items():
    # SVM and KNN benefit from scaling
    if name in ['SVM', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'accuracy': acc, 'roc_auc': auc, 'model': model}
    print(f'{name:20s} | Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}')

In [ ]:
# Visualise model comparison
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
aucs = [results[n]['roc_auc'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
ax.bar(x + width/2, aucs, width, label='ROC-AUC', color='salmon')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0.7, 1.0)
ax.set_title('Model Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (XGBoost & Random Forest)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- XGBoost ---
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
}
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
    xgb_params, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)
print('Best XGBoost params:', xgb_grid.best_params_)
print('Best XGBoost ROC-AUC (CV):', xgb_grid.best_score_)

In [ ]:
# --- Random Forest ---
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)
print('Best RF params:', rf_grid.best_params_)
print('Best RF ROC-AUC (CV):', rf_grid.best_score_)

In [ ]:
# Evaluate tuned models on test set
for name, grid in [('XGBoost (tuned)', xgb_grid), ('Random Forest (tuned)', rf_grid)]:
    y_pred = grid.predict(X_test)
    y_prob = grid.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f'{name:30s} | Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}')

## 8. Feature Importance (Best Model)

In [ ]:
best_xgb = xgb_grid.best_estimator_
importances = pd.Series(best_xgb.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(8, 6), color='steelblue')
plt.title('XGBoost Feature Importances')
plt.tight_layout()
plt.show()

## 9. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

for name, result in results.items():
    model = result['model']
    if name in ['SVM', 'KNN']:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = result['roc_auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Save Models & Scaler

### Output path note

Models are saved to **`models/bank/`** so they are immediately loadable by the production
FastAPI server (`api.py`) without any path adjustments.

This notebook produces **5 of the 7** models the API expects. The remaining two
(Gradient Boosting, Stacking) and the `schema.json` integrity file are produced by
`training/train_bank.py`. Run that script for a complete production-ready artifact set:

```bash
python training/train_bank.py
```

In [ ]:
# models/bank/ was created by the setup cell above
import sklearn, xgboost as xgb_lib

saves = {
    'xgboost_model.pkl':       xgb_grid.best_estimator_,
    'random_forest_model.pkl': rf_grid.best_estimator_,
    'decision_tree_model.pkl': results['Decision Tree']['model'],
    'svm_model.pkl':           results['SVM']['model'],
    'knn_model.pkl':           results['KNN']['model'],
    'scaler.pkl':              scaler,
}
import pickle
for fname, obj in saves.items():
    with open(MODELS_DIR / fname, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  saved {fname}')

# Write schema.json so the API's startup schema validation passes
import json as _json
schema = {
    'schema_version': 1,
    'feature_columns': list(X.columns),
    'n_features': int(X.shape[1]),
    'trained_with': {
        'scikit-learn': sklearn.__version__,
        'xgboost': xgb_lib.__version__,
        'numpy': np.__version__,
    },
}
with open(MODELS_DIR / 'schema.json', 'w') as f:
    _json.dump(schema, f, indent=2)
print(f'  saved schema.json')
print(f'\n5/7 models + scaler + schema saved to {MODELS_DIR}/')
print('Run training/train_bank.py for the full 7-model set.')